# 02 · Structured output and tool calling

Tutorial sections 4–5 ([`TUTORIAL.md`](../TUTORIAL.md)).

All models are reached through AWS Bedrock. Run the setup cell first: it loads `.env.local` next to this notebook (AWS profile, region, model choice, `LIVE` switch) and prints those settings. Credentials come from your AWS SSO profile, never from the notebook or the file. With `LIVE` unset or `0`, every call returns a mock response: no AWS access, no cost. See `00_credentials.ipynb` to set up.

`FAST`, `SMART` and `ALT` are the three Bedrock models from `.env.local` (defaults: Claude Haiku 5.5, Claude Opus 5.5, Amazon Nova 2 Lite).

In [1]:
from _env import ALT, FAST, LIVE, MODELS, SMART, mock, status  # loads .env.local; import before litellm

import litellm

litellm.suppress_debug_info = True
status()

env file    : .env.local (found)
mode        : LIVE (real Bedrock calls, real cost)
AWS_PROFILE : set (from .env.local)
AWS_REGION  : us-west-2
MODEL_FAST  : bedrock/us.anthropic.claude-haiku-5-5 (default)
MODEL_SMART : bedrock/us.anthropic.claude-opus-5-5 (default)
MODEL_ALT   : bedrock/us.amazon.nova-2-lite-v1:0 (default)


## 4. Structured output

Pass a Pydantic model as `response_format`, then validate the reply. [Section 4](../TUTORIAL.md#4-structured-output)

In [2]:
import json

from pydantic import BaseModel, ValidationError


class Invoice(BaseModel):
    vendor: str
    total: float
    currency: str
    line_items: list[str]


TEXT = """ACME Tools Ltd. Invoice #8812. 2x hammer (24.00), 1x saw (31.50).
Total due: 55.50 EUR."""
MODEL = FAST

print("supports json schema:", litellm.supports_response_schema(model=MODEL))

resp = litellm.completion(
    model=MODEL,
    messages=[{"role": "user", "content": f"Extract the invoice.\n\n{TEXT}"}],
    response_format=Invoice,
    max_tokens=2048,
    **mock(json.dumps({"vendor": "ACME Tools Ltd.", "total": 55.5, "currency": "EUR",
                       "line_items": ["2x hammer", "1x saw"]})),
)
invoice = Invoice.model_validate_json(resp.choices[0].message.content)
print(invoice)

try:
    Invoice.model_validate_json('{"vendor": "ACME", "total": "fifty"}')
except ValidationError as e:
    print(f"\nbad output is caught: {e.error_count()} errors, e.g. {e.errors()[0]['loc']}")

supports json schema: True
vendor='ACME Tools Ltd.' total=55.5 currency='EUR' line_items=['2x hammer (24.00)', '1x saw (31.50)']

bad output is caught: 3 errors, e.g. ('total',)


## 5. Tool calling

A complete loop in the OpenAI tool format; LiteLLM translates it to Bedrock's Converse tool format and back. [Section 5](../TUTORIAL.md#5-tool-calling)

In [3]:
MODEL = SMART
TOOLS = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
}]


def get_weather(city):
    return {"city": city, "temp_c": {"Paris": 14, "Tokyo": 21}.get(city, 18), "sky": "cloudy"}


FAKE_TURNS = [
    {"mock_tool_calls": [
        {"id": "call_1", "type": "function",
         "function": {"name": "get_weather", "arguments": json.dumps({"city": "Paris"})}},
        {"id": "call_2", "type": "function",
         "function": {"name": "get_weather", "arguments": json.dumps({"city": "Tokyo"})}},
    ]},
    {"mock_response": "Tokyo is warmer: 21°C versus 14°C in Paris, both cloudy."},
]

messages = [{"role": "user", "content": "Is it warmer in Paris or Tokyo right now?"}]
for turn in range(5):
    fake = {} if LIVE else FAKE_TURNS[turn]
    msg = litellm.completion(model=MODEL, messages=messages, tools=TOOLS, max_tokens=2048,
                             **fake).choices[0].message
    messages.append(msg)
    if not msg.tool_calls:
        print("final answer:", msg.content)
        break
    for call in msg.tool_calls:
        args = json.loads(call.function.arguments)
        result = get_weather(**args)
        print(f"turn {turn}: model called {call.function.name}({args}) -> {result}")
        messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})

turn 0: model called get_weather({'city': 'Paris'}) -> {'city': 'Paris', 'temp_c': 14, 'sky': 'cloudy'}
turn 0: model called get_weather({'city': 'Tokyo'}) -> {'city': 'Tokyo', 'temp_c': 21, 'sky': 'cloudy'}
final answer: Tokyo is warmer right now. It's 21°C (about 70°F) there and 14°C (about 57°F) in Paris, so Tokyo is 7°C warmer. Both cities are cloudy.
